# 01 · La canasta de los jubilados (ENGHo 2017/18)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/IPC_jubilados/blob/main/notebooks/01_canasta_jubilados.ipynb)

Arma las ponderaciones del IIJP a partir de los **microdatos** de la Encuesta Nacional de Gastos de los
Hogares 2017/18 (INDEC): gasto de consumo de cada hogar por división COICOP (`gc_01`…`gc_12`).

- Reproduce la canasta del TP original (hogares con al menos un mayor de 65, mezcla 0,69 / 0,31).
- Define la población de referencia del IIJP: **hogares jubilados** (jubilaciones y pensiones ≥ 50% del ingreso).
- Compara con el IPC oficial, que todavía usa la estructura de gasto de la **ENGHo 2004/05**.
- Mide la incertidumbre muestral de las ponderaciones (bootstrap de hogares).

In [ ]:
# Configuracion: en Colab clona (o actualiza) el repo; en local usa la carpeta del repo
import os, sys, subprocess
REPO = 'https://github.com/santiagoriverti/IPC_jubilados.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/IPC_jubilados'
    if os.path.exists(RAIZ):
        subprocess.run(['git', '-C', RAIZ, 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
sys.path.insert(0, RAIZ)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import pipeline, graficos, indices, ponderaciones as pond
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 30)

In [ ]:
N_BOOTSTRAP = 300   # replicas para los intervalos de confianza (0 = no calcular)
R = pipeline.calcular(n_bootstrap=N_BOOTSTRAP)

## Poblaciones

Cantidad de hogares en la muestra y expandidos (con el factor `pondera`) de cada definición.

In [ ]:
display(R['poblaciones'])
print(f"Proporcion ponderada de hogares con un solo mayor de 65: {R['peso_tp_1_mayor']:.4f} (el TP usa 0,69)")

## Ponderaciones por división (en %)

- `ipc_oficial`: IPC nacional del INDEC (ENGHo 2004/05).
- `tp_original`: lo que cargó el TP. `tp`: la misma canasta recalculada desde los microdatos (coincide).
- `total`: todos los hogares con la ENGHo 2017/18 → lo que sería un IPC con canasta actualizada.
- `jubilados`: **canasta del IIJP**. `jub_bajos` / `jub_altos`: mitades por ingreso per cápita.

In [ ]:
W = R['ponderaciones'].set_index('division')
display((W * 100).round(1))

In [ ]:
fig = graficos.g01_ponderaciones(R); plt.show()

**Lectura.** Frente a todos los hogares de la misma encuesta, los hogares jubilados gastan más en
alimentos (+3,7 p.p.), salud (+5 p.p.) y equipamiento, y menos en educación, transporte, restaurantes y
ropa. Vivienda y servicios pesa **lo mismo** (14,5%) en ambos: la mayor ponderación de vivienda que el TP
atribuía a los jubilados (13,7% vs 9,4% del IPC) es en realidad un efecto de la **canasta nueva**
(2017/18 vs 2004/05), no de la edad.

## Incertidumbre muestral

Intervalos de 95% por bootstrap (remuestreo de hogares estratificado por región). Son angostos: la
diferencia entre jubilados y el total de hogares en salud, educación o transporte no es ruido muestral.

In [ ]:
if 'bootstrap_ponderaciones' in R:
    b = R['bootstrap_ponderaciones'] * 100
    b.index = [pond.DIVISIONES_CORTO[c] for c in b.index]
    display(b.round(1))

## Sensibilidad: ponderaciones "democráticas"

El IPC usa ponderaciones plutocráticas (cada hogar pesa según cuánto gasta). Promediar las estructuras
de cada hogar (democráticas) da más peso a los hogares de menores ingresos.

In [ ]:
D = R['ponderaciones_democraticas'].set_index('division')
display((D[['total', 'jubilados', 'jub_bajos']] * 100).round(1))